# Lesson 6: Functions and recursion

**Why no sorting algorithm can beat n log n** · Intermediate · from [Hypnotic Patterns](https://hypnoticpatterns.doner.cloud)

Shuffle a deck and sort it by comparing two cards at a time. Some methods compare neighbours over and over; others split the deck in halves, then halves of halves. On the same shuffle the second kind finishes far sooner, and no method that only compares can do much better than that.

Watch the films and play with the live demo: [Sorting race](https://hypnoticpatterns.doner.cloud/pattern/sorting-race). The same lesson with a quiz: [https://hypnoticpatterns.doner.cloud/learn/sorting-race](https://hypnoticpatterns.doner.cloud/learn/sorting-race).

## What you will learn

- functions, and passing a function as a value
- recursion: a function that calls itself on smaller parts
- a counter kept in a dictionary
- assert to check a result
- how work grows: n² against n·log n

## The script

Run the cell below (Shift + Enter). numpy and matplotlib are already in Colab.

In [ ]:
"""Sorting race: four algorithms on the same shuffled list, counting their comparisons.
Run: python sorting-race.py   (needs: pip install numpy matplotlib)
"""
import math

import numpy as np
import matplotlib.pyplot as plt

plt.style.use("dark_background")

n = 100
data = list(np.random.default_rng().permutation(n))
count = {"comparisons": 0}


def less(a, b):  # every algorithm compares only through this, so we can count
    count["comparisons"] += 1
    return a < b


def bubble(xs):
    xs = xs[:]
    for end in range(len(xs) - 1, 0, -1):
        for i in range(end):
            if less(xs[i + 1], xs[i]):
                xs[i], xs[i + 1] = xs[i + 1], xs[i]
    return xs


def insertion(xs):
    xs = xs[:]
    for i in range(1, len(xs)):
        j = i
        while j > 0 and less(xs[j], xs[j - 1]):
            xs[j], xs[j - 1] = xs[j - 1], xs[j]
            j -= 1
    return xs


def merge(xs):
    if len(xs) < 2:
        return xs
    left, right = merge(xs[: len(xs) // 2]), merge(xs[len(xs) // 2 :])
    out = []
    while left and right:
        out.append(left.pop(0) if not less(right[0], left[0]) else right.pop(0))
    return out + left + right


def quick(xs):
    if len(xs) < 2:
        return xs
    pivot, rest = xs[0], xs[1:]
    smaller = [x for x in rest if less(x, pivot)]
    return quick(smaller) + [pivot] + quick([x for x in rest if x not in smaller])


results = {}
for name, sort in [("bubble", bubble), ("insertion", insertion), ("merge", merge), ("quick", quick)]:
    count["comparisons"] = 0
    assert sort(data) == sorted(data)
    results[name] = count["comparisons"]
print(results, "| at least log2(n!) =", math.ceil(math.log2(math.factorial(n))))

plt.figure(figsize=(7, 4))
plt.bar(list(results), list(results.values()), color=["#f0abfc", "#c4b5fd", "#5eead4", "#5eead4"])
plt.axhline(math.log2(math.factorial(n)), color="white", linestyle="--", label="log2(n!): no comparison sort does better")
plt.ylabel(f"comparisons for n = {n}")
plt.legend()
plt.show()

# Try: n = 1000 (bubble sort needs about n*n/2, merge sort about n*log2(n))

## Line by line

1. `def less(a, b):`  
   Every algorithm compares through this one function, so the comparisons can be counted in one place.
2. `count["comparisons"] += 1`  
   A dictionary holds the count, so the function can change it without the global keyword.
3. `def merge(xs):`  
   Merge sort: sort each half, then merge the two sorted halves by always taking the smaller front item.
4. `left, right = merge(xs[: len(xs) // 2]), merge(xs[len(xs) // 2 :])`  
   Recursion: merge calls itself on each half, until a part has fewer than two items.
5. `assert sort(data) == sorted(data)`  
   assert stops the script if a sort gets the order wrong: a test built into the program.
6. `math.ceil(math.log2(math.factorial(n)))`  
   The lower bound: log₂ of the number of possible orders, n!, rounded up.

## Try this

Set n = 1000: bubble sort’s count grows about a hundredfold, merge sort’s about sixteenfold.

## Exercises

Write your code in the empty cells. The hints and answers are at the end of the notebook.

1. For n = 100, which sort needs the fewest comparisons, and how close is it to the bound?
2. Why does this bubble sort always need n·(n − 1) ÷ 2 comparisons?
3. Give quick sort a list that is already sorted. What happens to its count?

**Exercise 1.** For n = 100, which sort needs the fewest comparisons, and how close is it to the bound?

In [ ]:
# your code here

**Exercise 2.** Why does this bubble sort always need n·(n − 1) ÷ 2 comparisons?

In [ ]:
# your code here

**Exercise 3.** Give quick sort a list that is already sorted. What happens to its count?

In [ ]:
# your code here

## Why it works

Bubble sort and insertion sort move items about one place at a time, so for n items they can need about n²/2 comparisons: for 1,000 items, around half a million.

Merge sort and quicksort split the job instead. There are about log₂ n levels of halving, each costing about n comparisons, so about n·log₂ n in all: for 1,000 items, around 10,000.

n items can arrive in n! different orders, and each comparison is a yes or no that at best halves the orders still possible. Telling n! orders apart therefore takes at least log₂(n!) comparisons, which grows like n·log₂ n. For four items there are 24 orders: four answers tell only 16 apart, so five are needed, and merge sort never needs more.

Sorts that do not compare, such as counting sort or radix sort, read the values themselves and can go faster on suitable data, as in the film’s chapter “Sorting Without Comparing”.

`comparisons ≥ log₂(n!) ≈ n·log₂ n − 1.44·n      bubble sort: up to n·(n − 1) ÷ 2`

## Hints and answers

1. *Hint:* Run the script; it prints the counts and the bound.  
   *Answer:* Merge sort, about 540 against a bound of 525. Bubble sort needs exactly 4,950, that is 100 · 99 ÷ 2.
2. *Hint:* Look at its two loops.  
   *Answer:* This version always runs both loops to the end: (n − 1) + (n − 2) + … + 1 = n·(n − 1) ÷ 2 comparisons, whatever the order.
3. *Hint:* data = list(range(n)).  
   *Answer:* It jumps to 4,950, as bad as bubble sort: with the first item as pivot, every split leaves one side empty.

## Sources

- [Sorting algorithm (Wikipedia)](https://en.wikipedia.org/wiki/Sorting_algorithm)
- [Comparison sort: the number of comparisons (Wikipedia)](https://en.wikipedia.org/wiki/Comparison_sort#Number_of_comparisons_required_to_sort_a_list)
- [Merge sort (Wikipedia)](https://en.wikipedia.org/wiki/Merge_sort)
- [Radix sort (Wikipedia)](https://en.wikipedia.org/wiki/Radix_sort)

Free to use for learning and in class. Made for the YouTube channel [Hypnotic Patterns](https://www.youtube.com/@HypnoticPatternsAudio).